<h1>CS152 Assignment 2: The 8-puzzle</h1>

Before you turn in this assignment, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then run the test cells for each of the questions you have answered.  Note that a grade of 3 for the A* implementation requires all tests in the "Basic Functionality" section to be passed.  The test cells pass if they execute with no errors (i.e. all the assertions are passed).

Make sure you fill in any place that says `YOUR CODE HERE`.  Be sure to remove the `raise NotImplementedError()` statements as you implement your code - these are simply there as a reminder if you forget to add code where it's needed.

---

<h1>
Question 1    
</h1>
Define your <code>PuzzleNode</code> class below.  Ensure that you include all attributes that you need to implement an A* search.  If you wish, you can even include member functions, such as a function to generate successor states.  Alternatively, you can code up this functionality later in the <code>solvePuzzle</code> function.

In [12]:
import heapq

#Now, define the class PuzzleNode:
class PuzzleNode:
    """
    Provides a structure for performing A* search. 

    """
    def __init__(self, state, parent=None, action=None, path_cost=0):
        """
        Initializes a PuzzleNode object.

        Input:
        May be a list of lists, so we need to convert to tuple of tuples for immutability and hashing.


        Returns:
        None
        """
        self.state = state
        self.parent = parent  # tile we came from
        self.action = action  #move that produced this state
        self.path_cost = path_cost # g(n) - the cost to reach this node from the start (number of moves)
        if isinstance(state, list):
            self.state = tuple(tuple(row) for row in state)
        else:
            self.state = state
    

    def get_successors(self):
        """
        Slides the blank in every valid direction and returns the resulting nodes.
        Each child inherits this node as parent and gets path_cost + 1.

        Returns:
        list: A list of PuzzleNode objects representing the successors.
        """
        successors = []
        # Find the position of the blank tile (0)
        blank_pos = [(i, row.index(0)) for i, row in enumerate(self.state) if 0 in row][0]
        x, y = blank_pos

        # Define possible action steps
        moves = {
            'down': (x + 1, y),
            'up': (x - 1, y),
            'right': (x, y + 1),
            'left': (x, y - 1)
        }

        for action, (new_x, new_y) in moves.items():
            if 0 <= new_x < len(self.state) and 0 <= new_y < len(self.state[0]):
                # Create a new state by swapping the blank tile with the adjacent tile
                new_state = [list(row) for row in self.state]  # Convert to list of lists for mutability
                
                
                new_state[x][y], new_state[new_x][new_y] = new_state[new_x][new_y], new_state[x][y]  # Swaps tiles
                successors.append(PuzzleNode(new_state, parent=self, action=action, path_cost=self.path_cost + 1))
        
        return successors

    def __eq__(self, other):
        """
        Checks if two PuzzleNode objects are equal based on their state.

        Returns:
        bool: True if the states are equal, False otherwise.
        """
        return self.state == other.state
    
    def __hash__(self):
        """
        Returns a hash value for the PuzzleNode based on its state. 
        
        """
        return hash(self.state)     


    def __str__(self): 
        """
        String representation of the puzzle state for visualizing it.
        
        """
        rows = []
        for row in self.state:
            rows.append("  ".join("_" if val == 0 else str(val) for val in row))
        return "\n".join(rows)


<h1>
Question 2    
</h1>
Define your heuristic functions using the templates below.  Ensure that you extend the <code>heuristics</code> list to include all the heuristic functions you implement.  Note that state will be given as a list of lists, so ensure your function accepts this format.  You may use packages like numpy if you wish within the functions themselves.

In [13]:
# Misplaced tiles heuristic
def h1(state):
    """
    This function returns the number of misplaced tiles, given the board state
    
    Counts tiles not yet in their goal position (excluding the 0).
    Admissible: each misplaced tile needs at least 1 move, so we never
    overestimate.
    
    Goal layout:  
        0 1 2
        3 4 5
        6 7 8
    """
    n = len(state)
    misplaced = 0
    for i in range(n):
        for j in range(n):
            if state[i][j] != 0 and state[i][j] != (i)*n +j: 
                misplaced += 1
    return misplaced

# Manhattan distance heuristic
def h2(state):
    """
    This function returns the Manhattan distance from the solved state, given the board state
    Input:
        state: the board state as a list of lists
    Output:
        h: the Manhattan distance
    
    Sums the row and column distances each tile needs to travel to its goal.
    Admissible and consistent — h2 >= h1 always, so it's a tighter bound
    and A* expands fewer nodes with this heuristic.

    """
    n = len(state)
    distance = 0
    for i in range(n):
        for j in range(n):
            tile = state[i][j]
            if tile != 0:  # Skip the blank tile
                # Calculate the goal position of the tile
                goal_x = (tile) % n
                goal_y = (tile) // n
                distance += abs(i - goal_y) + abs(j - goal_x)
    return distance
    
def h3(state):
    """
    Linear conflict that dominates Manhattan distance.
    Manhattan distance treats every tile as if it moves in isolation. 
    But I saw that when two tiles are both in their correct row but in the wrong order, they're physically blocking each other
    -- one has to leave the row, let the other pass, then come back. That's at least 2 extra moves per pair, which Manhattan distance never accounts for.
     Adding these up gives a tighter but still admissible heuristic.


    """
    n = len(state)
    conflict = 0

    # Row conflicts: both tiles belong in row i, but are in the wrong order
    for i in range(n):
        for j in range(n):
            tj = state[i][j]
            if tj == 0 or tj // n != i:
                continue
            for k in range(j + 1, n):
                tk = state[i][k]
                if tk == 0 or tk // n != i:
                    continue
                if tj > tk:  # tj is left of tk but should be right
                    conflict += 2

    # Column conflicts: both tiles belong in column j, but are in the wrong order
    for j in range(n):
        for i in range(n):
            tj = state[i][j]
            if tj == 0 or tj % n != j:
                continue
            for k in range(i + 1, n):
                tk = state[k][j]
                if tk == 0 or tk % n != j:
                    continue
                if tj > tk:
                    conflict += 2

    return h2(state) + conflict

heuristics = [h1, h2, h3]


<h1>
Question 3    
</h1>
Code up your A* search using the SolvePuzzle function within the template below.  Please do not modify the function header, otherwise the automated testing will fail.  You may define other functions or import packages as needed in this cell or by adding additional cells.

In [14]:
import heapq


# Main solvePuzzle function.
def solvePuzzle(state, heuristic):
    """This function should solve the n**2-1 puzzle for any n > 2.

    A* selects nodes like f(n) = g(n) + h(n): actual cost so far plus the
    heuristic estimate to goal. 
    f never decreases along any path, so the first time we expand a state it's guaranteed optimal.


    Inputs:
        state: The initial state of the puzzle
        heuristic: h1 or h2 as defined in Question 2.
    Outputs:
        steps: The number of steps to optimally solve the puzzle (excluding the initial state)
        exp(anded nodes): The number of nodes expanded to reach solution
        max_frontier: The maximum size of the frontier over the whole search
        opt_path: The optimal path. opt_path[:,:,i] should give a list of lists
                    that represents the state of the board at the ith step of the solution.

    """
    #Extension - checks whether the puzzle is solvable in the first place.

    def is_solvable(state):
        """
        if you take a solved puzzle and swap any two tiles, it becomes permanently unsolvable. This comes down to
        permutation parity: every move (sliding a tile) changes the puzzle's
        parity. If the start state has the wrong parity
        relative to the goal, no sequence of moves can ever fix it.
        
        We can use inversion, where a is before b in a sequence but a > b:
        If the # of inversions is even, the puzzle is solvable for odd n.
        If the # of inversions + blank row from bottom is odd, the puzzle is solvable for even n.
        """
        n = len(state)
        sequence = [tile for row in state for tile in row if tile != 0] # Flatten the board and to look at the sequence of tiles without the blank
        inversions = 0
        for i in range(len(sequence)):
            for j in range(i + 1, len(sequence)):
                if sequence[i] > sequence [j]:
                    inversions += 1
        if n % 2 == 1:
            return inversions % 2 == 0
        else:
            blank_row_from_bottom = n - next(i for i, row in enumerate(state) if 0 in row)
            return (inversions + blank_row_from_bottom) % 2 == 1
    
    
    ##############################

    # (a) Validate state - must have nxn shape and n^2-1 tiles.
    n = len(state)
    if n < 2:
        return 0, 0, 0, None, -1
    for row in state:
        if len(row) != n:
            return 0, 0, 0, None, -1
    
    if sorted(tile for row in state for tile in row) != list(range(n * n)): #for each tile of the board, checks if the sorted list matches the goal state's tiles
        return 0, 0, 0, None, -1

    # Goal: tile has a consistent row and column as its goal state
    goal = tuple(tuple(r * n + c for c in range(n)) for r in range(n)) 

    start_node = PuzzleNode(state)

    

    # Min-heap ordered. Counter breaks ties to avoid comparing PuzzleNodes.
    counter = 0
    
    frontier = []
    heapq.heappush(frontier, (heuristic([list(row) for row in start_node.state]), counter, start_node)) #frontier now contains initial node

    visited = {} # tracks the expanded states so we don't visit them twice
    exp = 0  # number of nodes expanded
    frontier_size = 1 # tracks the size of the frontier manually as qsize() is an estimate
    max_frontier = 1 # worst-case frontier size over the whole search

    while frontier:
        _, _, node = heapq.heappop(frontier)
        frontier_size -= 1

        # A state can appear in the frontier multiple times with different g values.
        # Skip it if we've already expanded it with other path.
        
        if node.state in visited:
            continue
        visited[node.state] = node.path_cost
        exp += 1


        # Goal test at expansion--if we checked at generation
        # we might accept a node that looks like the goal but was reached
        # with a suboptimal path, before a cheaper path was fully explored.
        
        if node.state == goal:
            path = []
            cur = node
            while cur is not None:
                path.append([list(row) for row in cur.state])
                cur = cur.parent
            path.reverse()
            return node.path_cost, exp, max_frontier, path, 0

        for child in node.get_successors():
            if child.state not in visited:
                f = child.path_cost + heuristic([list(row) for row in child.state])
                counter += 1
                heapq.heappush(frontier, (f, counter, child))
                frontier_size += 1
                if frontier_size > max_frontier:
                    max_frontier = frontier_size

    # Check solvability before searching, because an unsolvable puzzle would
    # cause A* to exhaust the entire state space with no success.
    if not is_solvable(state):
        return 0, 0, 0, None, -2
    
    return 0, 0, 0, None, -1


<h1>Extension Questions</h1>

The extensions can be implemented by modifying the code from Q2-3 above appropriately.

1. <b>Initial state solvability:</b>  Modify your SolvePuzzle function code in Q3 to return -2 if an initial state is not solvable to the goal state.
2. <b>Extra heuristic function:</b> Add another heuristic function (e.g. pattern database) that dominates the misplaced tiles and Manhattan distance heuristics to your Q2 code.
3. <b>Memoization:</b>  Modify your heuristic function definitions in Q2 by using a Python decorator to speed up heuristic function evaluation

There are test cells provided for extension questions 1 and 2.

<h1>Basic Functionality Tests</h1>
The cells below contain tests to verify that your code is working properly to be classified as basically functional.  Please note that a grade of <b>3</b> on #aicoding and #search as applicable for each test requires the test to be successfully passed.  <b>If you want to demonstrate some other aspect of your code, then feel free to add additional cells with test code and document what they do.<b>

In [15]:
## Test for state not correctly defined

incorrect_state = [[0,1,2],[2,3,4],[5,6,7]]
_,_,_,_,err = solvePuzzle(incorrect_state, lambda state: 0)
assert(err == -1)

In [16]:
## Heuristic function tests for misplaced tiles and manhattan distance

# Define the working initial states
working_initial_states_8_puzzle = ([[2,3,7],[1,8,0],[6,5,4]], [[7,0,8],[4,6,1],[5,3,2]], [[5,7,6],[2,4,3],[8,1,0]])

# Test the values returned by the heuristic functions
h_mt_vals = [7,8,7]
h_man_vals = [15,17,18]

for i in range(0,3):
    h_mt = heuristics[0](working_initial_states_8_puzzle[i])
    h_man = heuristics[1](working_initial_states_8_puzzle[i])
    assert(h_mt == h_mt_vals[i])
    assert(h_man == h_man_vals[i])


In [17]:
## A* Tests for 3 x 3 boards
## This test runs A* with both heuristics and ensures that the same optimal number of steps are found
## with each heuristic.

# Optimal path to the solution for the first 3 x 3 state
opt_path_soln = [[[2, 3, 7], [1, 8, 0], [6, 5, 4]], [[2, 3, 7], [1, 8, 4], [6, 5, 0]], 
                 [[2, 3, 7], [1, 8, 4], [6, 0, 5]], [[2, 3, 7], [1, 0, 4], [6, 8, 5]], 
                 [[2, 0, 7], [1, 3, 4], [6, 8, 5]], [[0, 2, 7], [1, 3, 4], [6, 8, 5]], 
                 [[1, 2, 7], [0, 3, 4], [6, 8, 5]], [[1, 2, 7], [3, 0, 4], [6, 8, 5]], 
                 [[1, 2, 7], [3, 4, 0], [6, 8, 5]], [[1, 2, 0], [3, 4, 7], [6, 8, 5]], 
                 [[1, 0, 2], [3, 4, 7], [6, 8, 5]], [[1, 4, 2], [3, 0, 7], [6, 8, 5]], 
                 [[1, 4, 2], [3, 7, 0], [6, 8, 5]], [[1, 4, 2], [3, 7, 5], [6, 8, 0]], 
                 [[1, 4, 2], [3, 7, 5], [6, 0, 8]], [[1, 4, 2], [3, 0, 5], [6, 7, 8]], 
                 [[1, 0, 2], [3, 4, 5], [6, 7, 8]], [[0, 1, 2], [3, 4, 5], [6, 7, 8]]]

astar_steps = [17, 25, 28]
for i in range(0,3):
    steps_mt, expansions_mt, _, opt_path_mt, _ = solvePuzzle(working_initial_states_8_puzzle[i], heuristics[0])
    steps_man, expansions_man, _, opt_path_man, _ = solvePuzzle(working_initial_states_8_puzzle[i], heuristics[1])
    # Test whether the number of optimal steps is correct and the same
    assert(steps_mt == steps_man == astar_steps[i])
    # Test whether or not the manhattan distance dominates the misplaced tiles heuristic in every case
    assert(expansions_man < expansions_mt)
    # For the first state, test that the optimal path is the same
    if i == 0:
        assert(opt_path_mt == opt_path_soln)


In [18]:
## A* Test for 4 x 4 board
## This test runs A* with both heuristics and ensures that the same optimal number of steps are found
## with each heuristic.

working_initial_state_15_puzzle = [[1,2,6,3],[0,9,5,7],[4,13,10,11],[8,12,14,15]]
steps_mt, expansions_mt, _, _, _ = solvePuzzle(working_initial_state_15_puzzle, heuristics[0])
steps_man, expansions_man, _, _, _ = solvePuzzle(working_initial_state_15_puzzle, heuristics[1])
# Test whether the number of optimal steps is correct and the same
assert(steps_mt == steps_man == 9)
# Test whether or not the manhattan distance dominates the misplaced tiles heuristic in every case
assert(expansions_mt >= expansions_man)

<h1>Extension Tests</h1>
The cells below can be used to test the extension questions.  Memoization if implemented will be tested on the final submission - you can test it yourself by testing the execution time of the heuristic functions with and without it.

In [19]:
## Puzzle solvability test

unsolvable_initial_state = [[7,5,6],[2,4,3],[8,1,0]]
_,_,_,_,err = solvePuzzle(unsolvable_initial_state, lambda state: 0)
assert(err == -2)

In [20]:
## Extra heuristic function test.  
## This tests that for all initial conditions, the new heuristic dominates over the manhattan distance.

dom = 0
for i in range(0,3):
    steps_new, expansions_new, _, _, _ = solvePuzzle(working_initial_states_8_puzzle[i], heuristics[2])
    steps_man, expansions_man, _, _, _ = solvePuzzle(working_initial_states_8_puzzle[i], heuristics[1])
    # Test whether the number of optimal steps is correct and the same
    assert(steps_new == steps_man == astar_steps[i])
    # Test whether or not the manhattan distance is dominated by the new heuristic in every case, by checking
    # the number of nodes expanded
    dom = expansions_man - expansions_new
    assert(dom > 0)

In [21]:
'''
AI Statement.

I made sure to understand and write all the code myself, and only used Claude for brainstorming and debugging help. Its answers prompted me
to think about edge cases and implementation details that I might have missed otherwise, but I made sure to write and understand all code myself.


I used Claude AI to help me with the following:
1. Brainstorm needed methods for the PuzzleNode class apart. It helped me identify the __hash__ method
as useful, as well as the get_successors method without giving me the code.
2. After giving it my initial mapping of the class, it helped me address gaps s.a. the way moves are defined,
copying the state before swapping for each successor, and passing the right arguments. No code was generated.
3. Slight debugging due to indentation issues.
4. It helped me identify the need to check when a node is visited.
5. After an Assertion Error it helped me decode, I realized I have to push a node into the frontier.
6. Helped me brainstorm drawbacks of the manhattan distance heuristic, which prompted me to implement the third heuristic.
7. While defining the third heuristic, I asked Claude to break down the inversion count parity testing for whether a puzzle is solvable.

During the working process, I started with brainstorming all ideas and asking Claude, after which I implemented them.

Here is the conversation: https://claude.ai/share/05b3d6a8-ef56-463d-a31e-44200722a129 

'''

'\nAI Statement.\n\nI made sure to understand and write all the code myself, and only used Claude for brainstorming and debugging help. Its answers prompted me\nto think about edge cases and implementation details that I might have missed otherwise, but I made sure to write and understand all code myself.\n\n\nI used Claude AI to help me with the following:\n1. Brainstorm needed methods for the PuzzleNode class apart. It helped me identify the __hash__ method\nas useful, as well as the get_successors method without giving me the code.\n2. After giving it my initial mapping of the class, it helped me address gaps s.a. the way moves are defined,\ncopying the state before swapping for each successor, and passing the right arguments. No code was generated.\n3. Slight debugging due to indentation issues.\n4. It helped me identify the need to check when a node is visited.\n5. After an Assertion Error it helped me decode, I realized I have to push a node into the frontier.\n6. Helped me brain

In [22]:
## Memoization test - will be carried out after submission